In [ ]:
def validate_scenario_scaling(scenario_dict, feature_ranges, feature_importance_df, scenario_name, pollutant='PM25'):
    """
    Validate if scenario values are appropriately scaled relative to importance and ranges
    """
    # Convert scenario dict to DataFrame
    scenario_df = pd.DataFrame(list(scenario_dict.items()), columns=['Feature', scenario_name])

    # Rename the feature column in feature_ranges
    feature_ranges_clean = feature_ranges.rename(columns={'Unnamed: 0': 'Feature'})

    # Merge all data
    validation_df = scenario_df.merge(feature_ranges_clean, on='Feature', how='inner')
    validation_df = validation_df.merge(feature_importance_df, on='Feature', how='inner')

    # Select the correct importance column based on pollutant
    importance_col = f'Importance_{pollutant}'
    validation_df = validation_df.rename(columns={importance_col: 'importance'})

    # Calculate normalized metrics
    validation_df['scenario_normalized'] = (validation_df[scenario_name] - validation_df['min']) / (validation_df['max'] - validation_df['min'])
    validation_df['importance_normalized'] = validation_df['importance'] / validation_df['importance'].max()

    # Calculate alignment score
    validation_df['alignment_score'] = validation_df['scenario_normalized'] * validation_df['importance_normalized']

    # Calculate if values are within realistic bounds
    validation_df['within_range'] = (validation_df[scenario_name] >= validation_df['min']) & (validation_df[scenario_name] <= validation_df['max'])

    return validation_df

def create_validation_report(scenario_dicts, feature_ranges, feature_importance_df, pollutants=['PM25', 'PM10', 'NO2']):
    """
    Create comprehensive validation report for all scenarios and pollutants
    """
    reports = {}

    for scenario_name, scenario_dict in scenario_dicts.items():
        scenario_reports = {}

        for pollutant in pollutants:
            # Validate for each pollutant
            validation_df = validate_scenario_scaling(
                scenario_dict, feature_ranges, feature_importance_df, scenario_name, pollutant
            )

            # Calculate summary statistics
            summary = {
                'total_features': len(validation_df),
                'features_within_range': validation_df['within_range'].sum(),
                'avg_alignment_score': validation_df['alignment_score'].mean(),
                'median_alignment_score': validation_df['alignment_score'].median(),
                'high_importance_alignment': validation_df.nlargest(10, 'importance')['alignment_score'].mean(),
                'coverage_quality': validation_df['alignment_score'].std(),  # Lower = more consistent
                'avg_scenario_normalized': validation_df['scenario_normalized'].mean(),
                'avg_importance_normalized': validation_df['importance_normalized'].mean()
            }

            scenario_reports[pollutant] = {
                'validation_df': validation_df,
                'summary': summary
            }

        reports[scenario_name] = scenario_reports

    return reports

# Run the validation
scenario_dicts = {
    'EU': eu_dict,
    'Swiss': swiss_dict,
    'Worldwide': worldwide_dict
}

validation_reports = create_validation_report(scenario_dicts, feature_ranges, feature_importance)

# Print summary results
print("SCENARIO VALIDATION SUMMARY")
print("=" * 60)

for scenario_name, reports in validation_reports.items():
    print(f"\n{scenario_name.upper()} SCENARIO:")
    print("-" * 40)

    for pollutant, report in reports.items():
        summary = report['summary']
        validation_df = report['validation_df']

        print(f"{pollutant}:")
        print(f"  Features analyzed: {summary['total_features']}")
        print(f"  Features within range: {summary['features_within_range']}/{summary['total_features']} ({summary['features_within_range']/summary['total_features']*100:.1f}%)")
        print(f"  Avg alignment score: {summary['avg_alignment_score']:.3f}")
        print(f"  High-importance feature alignment: {summary['high_importance_alignment']:.3f}")
        print(f"  Avg scenario value (normalized): {summary['avg_scenario_normalized']:.3f}")
        print(f"  Consistency (std): {summary['coverage_quality']:.3f}")

        # Show features that are out of range
        out_of_range = validation_df[~validation_df['within_range']]
        if len(out_of_range) > 0:
            print(f"  ⚠️  Features OUT OF RANGE: {list(out_of_range['Feature'])}")

In [ ]:
def scale_to_training_range(realistic_values, training_means, training_stds, training_mins, training_maxs):
    """
    Scale realistic values to match the training data distribution.
    Assumes the training data was standardized (mean=0, std=1) or normalized.
    """
    scaled_values = {}

    for feature, realistic_value in realistic_values.items():
        if feature in training_means.index:
            # Check if data was standardized (mean near 0, std near 1)
            if abs(training_means[feature]) < 10 and training_stds[feature] > 0.1:
                # Standardization scaling: (x - mean) / std
                scaled_value = (realistic_value - training_means[feature]) / training_stds[feature]
            else:
                # Min-max scaling or other normalization - use robust scaling
                # Since we don't know the original scaling, use z-score with training stats
                scaled_value = (realistic_value - training_means[feature]) / training_stds[feature]

            scaled_values[feature] = scaled_value
        else:
            print(f"Warning: {feature} not found in training data")
            scaled_values[feature] = realistic_value

    return scaled_values

def inverse_scale_from_training(scaled_values, training_means, training_stds):
    """
    Convert scaled values back to original units for interpretation.
    """
    original_values = {}

    for feature, scaled_value in scaled_values.items():
        if feature in training_means.index:
            original_value = (scaled_value * training_stds[feature]) + training_means[feature]
            original_values[feature] = original_value
        else:
            original_values[feature] = scaled_value

    return original_values

In [ ]:
# Check if there are major scaling issues in your scenario values
print("\nDATA QUALITY CHECK")
print("=" * 50)

for scenario_name, scenario_dict in scenario_dicts.items():
    print(f"\n{scenario_name}:")
    # Check a few key features
    key_features = ['Bus_age_Total', 'Car_age_Total', 'GDP per capita', 'Population']

    for feature in key_features:
        if feature in scenario_dict:
            # Get the range for this feature
            feature_range = feature_ranges[feature_ranges['Unnamed: 0'] == feature]
            if not feature_range.empty:
                min_val = feature_range['min'].iloc[0]
                max_val = feature_range['max'].iloc[0]
                scenario_val = scenario_dict[feature]

                normalized = (scenario_val - min_val) / (max_val - min_val) if max_val > min_val else 0
                print(f"  {feature}: {scenario_val} (normalized: {normalized:.3f})")

                if scenario_val < min_val or scenario_val > max_val:
                    print(f"    ⚠️  OUT OF RANGE! Should be between {min_val:.2f} and {max_val:.2f}")

In [ ]:
# Load your data
feature_ranges = pd.read_csv('feature_ranges.csv')
feature_importance = pd.read_csv('combined_feature_importances.csv')

# Your scenario dictionaries (assuming they're already loaded)
scenario_dicts = {
    'EU': eu_dict,
    'Swiss': swiss_dict,
    'Worldwide': worldwide_dict
}

# Run comprehensive validation
validation_reports = create_validation_report(scenario_dicts, feature_ranges, feature_importance)

# Print summary results
print("SCENARIO VALIDATION SUMMARY")
print("=" * 60)

for scenario_name, reports in validation_reports.items():
    print(f"\n{scenario_name.upper()} SCENARIO:")
    print("-" * 40)

    for pollutant, report in reports.items():
        summary = report['summary']
        print(f"{pollutant}:")
        print(f"  Features within range: {summary['features_within_range']}/{summary['total_features']}")
        print(f"  Avg alignment score: {summary['avg_alignment_score']:.3f}")
        print(f"  High-importance feature alignment: {summary['high_importance_alignment']:.3f}")
        print(f"  Consistency (std): {summary['coverage_quality']:.3f}")

Gunzgen predictions inference to other polutants through country local model

In [ ]:
import pandas as pd

def create_yearly_inputs(base_values, projection_funcs, start_year=2020, end_year=2050):
    """
    Create a dict of dynamic inputs for each year.

    Parameters
    ----------
    base_values : dict
        Base feature values for the starting year.
    projection_funcs : dict
        Dictionary of projection functions keyed by feature names.
    start_year : int
        First year for projections.
    end_year : int
        Last year for projections.

    Returns
    -------
    yearly_inputs : dict
        Dictionary keyed by year, each containing the updated features.
    """
    yearly_inputs = {}
    for year in range(start_year, end_year + 1):
        year_input = base_values.copy()
        for feature, func in projection_funcs.items():
            if feature in base_values:
                year_input[feature] = func(base_values[feature], year, start_year)
        yearly_inputs[year] = year_input
    return yearly_inputs

# Example usage:
sw_base_values = example_input.copy()  # your static base input dict
sw_funcs = get_swiss_projection_functions()
dynamic_inputs_sw = create_yearly_inputs(sw_base_values, sw_funcs, 2020, 2050)

# Now generate predictions for Gunzgen
years = list(range(2020, 2051))
predictions_gunzgen = []

for year in years:
    year_input = dynamic_inputs_sw[year]
    pred, _ = predict_with_country(
        xgb_reg_pm10_country_load,
        year_input,
        country_name="Switzerland",
        training_columns=training_columns
    )
    predictions_gunzgen.append(pred)


# Option 2: Rename with rename()
#pred_ts_gunzgen.name='pm10'
# Convert Series to DataFrame with column name 'pm10'
#if isinstance(pred_ts_gunzgen, pd.Series):
    #pred_ts_gunzgen = pred_ts_gunzgen.to_frame(name='pm10')
#pred_ts_gunzgen_full = infer_pm25_no2(pred_ts_gunzgen)

In [ ]:
# Define forecast horizon explicitly for VAR
forecast_index_sw = pd.date_range(start="2020", end="2050", freq="Y")
steps_sw = len(forecast_index_sw)  # 31 years
var_model_sw = VAR(tssw_diff_aligned)
lag_order_sw = var_model_sw.select_order(maxlags=5)
var_fitted_sw = var_model_sw.fit(lag_order_sw.aic)

forecast_mean_sw, lower_sw, upper_sw = var_fitted_sw.forecast_interval(
    tssw_diff_aligned.values[-lag_order_sw.aic:], steps=steps_sw
)
